# 02 ? Prepara??o dos dados
Continua??o da EDA: valida??o do CSV, divis?o por utilizador, sele??o expl?cita de entradas e prepara??o ajustada apenas ao treino.

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/cybercrime_forensic_dataset.csv').exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.dados import carregar_dados, dividir_dados, resumo_divisao
df = carregar_dados()
print("Dimens?o:", df.shape)
df.info()
display(df.isna().sum().to_frame("Ausentes"))
display(df.nunique().to_frame("Valores distintos"))
print("Duplicados completos:", df.duplicated().sum())
display(df["Label"].value_counts())

Dimens?o: (7400, 11)
<class 'pandas.DataFrame'>
RangeIndex: 7400 entries, 0 to 7399
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Timestamp          7400 non-null   str    
 1   User_ID            7400 non-null   int64  
 2   IP_Address         7400 non-null   str    
 3   Activity_Type      7400 non-null   str    
 4   Resource_Accessed  7400 non-null   str    
 5   File_Name          3196 non-null   str    
 6   Action             7400 non-null   str    
 7   Login_Attempts     2105 non-null   float64
 8   File_Size          3196 non-null   float64
 9   Anomaly_Type       7400 non-null   str    
 10  Label              7400 non-null   str    
dtypes: float64(2), int64(1), str(8)
memory usage: 636.1 KB


,Ausentes
Timestamp,0
User_ID,0
IP_Address,0
Activity_Type,0
Resource_Accessed,0
File_Name,4204
Action,0
Login_Attempts,5295
File_Size,4204
Anomaly_Type,0


,Valores distintos
Timestamp,7400
User_ID,5031
IP_Address,7353
Activity_Type,7
Resource_Accessed,36
File_Name,6
Action,7
Login_Attempts,10
File_Size,2382
Anomaly_Type,5


Duplicados completos: 0


Label
Normal        6167
Suspicious    1233
Name: count, dtype: int64

## Divis?o por utilizador
Existem v?rios registos do mesmo utilizador. O objetivo experimental fica definido como classificar atividades de **utilizadores n?o observados no treino**, dentro da popula??o e per?odo simulados. GroupShuffleSplit, com semente 42, reserva 20% dos utilizadores para teste; dos restantes, 25% ficam para valida??o (aproximadamente 60%/20%/20% dos utilizadores no total).
As percentagens de linhas e classes podem diferir: n?o se estratificam os r?tulos nem se procura uma divis?o com melhores resultados. A semente n?o ser? ajustada em fun??o das m?tricas.
Uma divis?o por linhas permitiria utilizadores repetidos nos conjuntos; uma divis?o temporal responderia ? previs?o de per?odos futuros, que n?o ? a alega??o desta experi?ncia. N?o existe um identificador de incidente para garantir a separa??o de acontecimentos relacionados.
O teste j? foi abrangido pela EDA original; essa limita??o ? reconhecida. As contagens seguintes verificam apenas a composi??o e n?o ser?o utilizadas para selecionar modelos.

In [2]:
conjuntos = dividir_dados(df)
display(resumo_divisao(conjuntos))
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    assert set(conjuntos[a]["User_ID"]).isdisjoint(conjuntos[b]["User_ID"])
    assert set(conjuntos[a].index).isdisjoint(conjuntos[b].index)
assert sum(len(parte) for parte in conjuntos.values()) == len(df)
print("Divis?o congelada: sem sobreposi??o de linhas ou utilizadores.")

,Registos,Utilizadores,Normal,Suspicious,% Suspicious
Conjunto,,,,,
treino,4416,3018,3682,734,16.62
validacao,1478,1006,1237,241,16.31
teste,1506,1007,1248,258,17.13


Divis?o congelada: sem sobreposi??o de linhas ou utilizadores.


## Decis?es sobre as caracter?sticas
- User_ID fica apenas como grupo de divis?o, n?o como entrada. A separa??o por utilizador impede memorizar a sua identidade entre treino e avalia??o.
- IP_Address ? quase ?nico, sem topologia ou contexto de rede confirmado; exclui-se para evitar memorizar endere?os sem significado transfer?vel.
- Resource_Accessed ? conservado como categoria. File_Name coincide com o ?ltimo componente do recurso nos registos em que existe; exclui-se a coluna redundante, sem perder essa informa??o.
- Timestamp fornece seno e cosseno da hora e do dia da semana: representam a continuidade dos ciclos sem usar o instante ?nico como identificador.
- Activity_Type, Action, Login_Attempts e File_Size s?o conservados. A previs?o refere-se a uma atividade j? registada, n?o ? antecipa??o do seu resultado.
- Label, Label_bin e Anomaly_Type s?o explicitamente proibidos nas entradas.
## Aus?ncias, escala e categorias
Os n?meros estruturalmente n?o aplic?veis tornam-se zero; um n?mero desconhecido numa atividade aplic?vel ? recusado para n?o o confundir com aus?ncia estrutural. File_Name n?o precisa de imputa??o por ter sido exclu?do. As categorias restantes ausentes usam Unknown e categorias novas s?o aceites pelo OneHotEncoder sem reaprender categorias.
A escala e a codifica??o s?o ajustadas exclusivamente no treino e, posteriormente, dentro de cada fold. O StandardScaler ? relevante para regress?o log?stica, KNN e SVM; a escala comum n?o ? necess?ria ?s ?rvores, mas mant?m a representa??o compar?vel. N?o s?o eliminadas linhas por aus?ncia estrutural nem por um crit?rio autom?tico de outliers. Na EDA, n?o foram identificados valores fora de 1,5 ? IQR; anomalias de contexto permanecem poss?veis.

In [3]:
import numpy as np
from src.preparacao import selecionar_entradas, obter_alvo, criar_preparacao, PROIBIDAS
X_treino = selecionar_entradas(conjuntos["treino"])
X_validacao = selecionar_entradas(conjuntos["validacao"])
y_treino = obter_alvo(conjuntos["treino"])
assert not (PROIBIDAS & set(X_treino.columns))
preparacao = criar_preparacao()
matriz_treino = preparacao.fit_transform(X_treino, y_treino)
matriz_validacao = preparacao.transform(X_validacao)
assert np.isfinite(matriz_treino).all() and np.isfinite(matriz_validacao).all()
print("Entradas permitidas:", list(X_treino.columns))
print("Matrizes treino/valida??o:", matriz_treino.shape, matriz_validacao.shape)
print("N?mero de caracter?sticas transformadas:", len(preparacao.get_feature_names_out()))
print("O teste n?o foi utilizado para ajustar ou comparar transforma??es.")

Entradas permitidas: ['Timestamp', 'Activity_Type', 'Resource_Accessed', 'Action', 'Login_Attempts', 'File_Size']
Matrizes treino/valida??o: (4416, 56) (1478, 56)
N?mero de caracter?sticas transformadas: 56
O teste n?o foi utilizado para ajustar ou comparar transforma??es.


## Desequil?brio
A distribui??o natural ? preservada em todos os conjuntos. Ser?o comparadas refer?ncias simples e, nos algoritmos que o suportam, pondera??o das classes. N?o ser? aplicada reamostragem global. Precision, Recall, F1 e Average Precision complementam a accuracy.
**Pr?xima etapa:** refer?ncias e compara??o no notebook 03. A prepara??o comum integra cada Pipeline de treino para impedir ajuste fora dos folds.